In [ ]:
import numpy as np
import torch
from IPython.display import HTML
from datasets import load_dataset

from autosim.utils import plot_spatiotemporal_video

In [ ]:
print("Downloading/Loading PDEBench 2D SWE dataset from Hugging Face...")
# This will download the dataset and cache it locally
dataset = load_dataset("PhysArena/PDEBench_2D_SWE", split="train")

In [ ]:
def get_trajectory(data_idx: int, channel_idx: int = 0) -> np.ndarray:
    sample = dataset[data_idx]
    data_key = "u" if "u" in sample else next(iter(sample.keys()))
    arr = np.array(sample[data_key])

    # Handle common layouts: (T, X, Y), (T, C, X, Y), (T, X, Y, C), or flat
    if arr.ndim == 3:
        trajectory = arr
    elif arr.ndim == 4:
        if arr.shape[1] <= 8 and arr.shape[2] > 8 and arr.shape[3] > 8:
            trajectory = arr[:, channel_idx, :, :]
        else:
            trajectory = arr[:, :, :, channel_idx]
    elif arr.ndim == 1 and arr.size == 101 * 128 * 128:
        trajectory = arr.reshape(101, 128, 128)
    else:
        raise ValueError(f"Unsupported trajectory shape: {arr.shape}")

    return trajectory.astype(np.float32)


def to_video_tensor(trajectory: np.ndarray) -> torch.Tensor:
    # plot_spatiotemporal_video expects (B, T, W, H, C)
    return torch.from_numpy(trajectory).unsqueeze(0).unsqueeze(-1)

In [ ]:
def animate_trajectory(trajectory: np.ndarray, title: str = "SWE trajectory"):
    video = to_video_tensor(trajectory)
    anim = plot_spatiotemporal_video(
        video,
        batch_idx=0,
        cmap="viridis",
        colorbar_mode="row",
        channel_names=["fluid height h"],
        title=title,
    )
    return anim

In [ ]:
traj = get_trajectory(250, channel_idx=0)
anim = animate_trajectory(traj, title="SWE trajectory (sample 250)")
HTML(anim.to_jshtml())

In [ ]:
traj = get_trajectory(100, channel_idx=0)
anim = animate_trajectory(traj, title="SWE trajectory (sample 100)")
HTML(anim.to_jshtml())

In [ ]:
print(f"Dataset loaded successfully! Total trajectories: {len(dataset)}")
sample = dataset[0]
print(f"Available keys in sample 0: {list(sample.keys())}")

traj0 = get_trajectory(0, channel_idx=0)
print("Resolved trajectory shape:", traj0.shape)

In [ ]:
anim = animate_trajectory(traj0, title="SWE trajectory (sample 0)")
HTML(anim.to_jshtml())